<a href="https://colab.research.google.com/github/livecow/livecow/blob/main/Colab_%EC%8B%9C%EC%9E%91%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
import sqlite3
import os

DB_NAME = "survey_app.db"

# Remove existing db file if it exists to start fresh
if os.path.exists(DB_NAME):
    os.remove(DB_NAME)
    print(f"Removed existing {DB_NAME}")

# Connect to a file-based SQLite database
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# SQL commands to create tables
sql_commands = """
CREATE TABLE users (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT,
    role TEXT CHECK(role IN ('조사원','중간관리자','총괄관리자'))
);

CREATE TABLE respondents (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    encrypted_name TEXT,
    encrypted_contact TEXT
);

CREATE TABLE gifts (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    respondent_id INTEGER,
    surveyor_id INTEGER,
    given_at DATETIME,
    FOREIGN KEY(respondent_id) REFERENCES respondents(id),
    FOREIGN KEY(surveyor_id) REFERENCES users(id)
);

CREATE TABLE inventory (
    surveyor_id INTEGER,
    total_items INTEGER,
    remaining_items INTEGER,
    FOREIGN KEY(surveyor_id) REFERENCES users(id)
);
"""

# Execute the SQL commands
cursor.executescript(sql_commands)

# Insert sample data for testing
cursor.execute("INSERT INTO users (username, role) VALUES (?, ?)", ('test_surveyor', '조사원'))
surveyor_id = cursor.lastrowid
cursor.execute("INSERT INTO respondents (encrypted_name, encrypted_contact) VALUES (?, ?)", ('encrypted_name_1', 'encrypted_contact_1'))
respondent_id = cursor.lastrowid
cursor.execute("INSERT INTO inventory (surveyor_id, total_items, remaining_items) VALUES (?, ?, ?)", (surveyor_id, 10, 10))

conn.commit()
conn.close()

print(f"Database tables created and sample data inserted successfully into {DB_NAME}.")

Removed existing survey_app.db
Database tables created and sample data inserted successfully into survey_app.db.


In [39]:
import subprocess
import time
import requests
import threading

# Change directory to where the 'app' executable is located
%cd /content/dist

# Function to run the Flask app in a separate thread
def run_flask_app():
    # Use subprocess.Popen to run the app in the background
    # stdout and stderr are redirected to avoid blocking the Colab kernel output
    global flask_process
    flask_process = subprocess.Popen(['./app'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    # Optionally, you can read output from the process if needed for debugging
    # for line in flask_process.stdout:
    #     print(f"Flask App Output: {line.decode().strip()}")

# Start the Flask app in a new thread
flask_thread = threading.Thread(target=run_flask_app)
flask_thread.start()

print("Flask app started in the background. Waiting for it to initialize...")
time.sleep(5) # Give the server a few seconds to start up

# Base URL for the Flask application
BASE_URL = "http://127.0.0.1:5000"

# Test the /give_gift endpoint
print("\n--- Testing /give_gift endpoint ---")
gift_data = {
    "respondent_id": 1,
    "surveyor_id": 1
}
try:
    response = requests.post(f"{BASE_URL}/give_gift", json=gift_data)
    print("Give Gift Response Status:", response.status_code)
    print("Give Gift Response Body:", response.json())
except requests.exceptions.ConnectionError as e:
    print(f"Error connecting to Flask app: {e}")
    print("Please ensure the Flask app started successfully.")

# Test the /inventory/<surveyor_id> endpoint
print("\n--- Testing /inventory/<surveyor_id> endpoint ---")
surveyor_id = 1
try:
    response = requests.get(f"{BASE_URL}/inventory/{surveyor_id}")
    print("Inventory Response Status:", response.status_code)
    print("Inventory Response Body:", response.json())
except requests.exceptions.ConnectionError as e:
    print(f"Error connecting to Flask app: {e}")
    print("Please ensure the Flask app started successfully.")

# Clean up: Terminate the Flask app process
if 'flask_process' in globals() and flask_process.poll() is None:
    flask_process.terminate()
    flask_process.wait()
    print("\nFlask app process terminated.")

print("API calls demonstration complete.")

/content/dist
Flask app started in the background. Waiting for it to initialize...

--- Testing /give_gift endpoint ---
Error connecting to Flask app: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /give_gift (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7db0c24038c0>: Failed to establish a new connection: [Errno 111] Connection refused'))
Please ensure the Flask app started successfully.

--- Testing /inventory/<surveyor_id> endpoint ---
Error connecting to Flask app: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /inventory/1 (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7db0c22be840>: Failed to establish a new connection: [Errno 111] Connection refused'))
Please ensure the Flask app started successfully.

Flask app process terminated.
API calls demonstration complete.


In [41]:
from flask import Flask, request, jsonify
import sqlite3
from datetime import datetime
import os

app = Flask(__name__)

def db_conn():
    # Ensure the database file is in the expected location or handle its creation
    # For simplicity, we'll use an in-memory db here, but if app.py is run standalone,
    # it would create a file-based db. Let's make it file-based for persistence
    return sqlite3.connect("survey_app.db")

@app.route("/give_gift", methods=["POST"])
def give_gift():
    data = request.json
    respondent_id = data["respondent_id"]
    surveyor_id = data["surveyor_id"]

    conn = db_conn()
    cur = conn.cursor()

    # 지급 로그 기록
    cur.execute("INSERT INTO gifts (respondent_id, surveyor_id, given_at) VALUES (?, ?, ?)",
                (respondent_id, surveyor_id, datetime.now()))

    # 재고 차감
    cur.execute("UPDATE inventory SET remaining_items = remaining_items - 1 WHERE surveyor_id = ?", (surveyor_id,))
    conn.commit()
    conn.close()

    return jsonify({"status": "success", "message": "답례품 지급 완료!"})

@app.route("/inventory/<int:surveyor_id>", methods=["GET"])
def inventory_status(surveyor_id):
    conn = db_conn()
    cur = conn.cursor()
    cur.execute("SELECT total_items, remaining_items FROM inventory WHERE surveyor_id=?", (surveyor_id,))
    result = cur.fetchone()
    conn.close()
    if result:
        return jsonify({"total": result[0], "remaining": result[1]})
    else:
        return jsonify({"error": "Surveyor not found or no inventory data"}), 404

# Save the Flask app code to app.py file
app_code = """
from flask import Flask, request, jsonify
import sqlite3
from datetime import datetime

app = Flask(__name__)

def db_conn():
    # Use a file-based database for persistence when packaged
    return sqlite3.connect("survey_app.db")

@app.route("/give_gift", methods=["POST"])
def give_gift():
    data = request.json
    respondent_id = data["respondent_id"]
    surveyor_id = data["surveyor_id"]

    conn = db_conn()
    cur = conn.cursor()

    cur.execute("INSERT INTO gifts (respondent_id, surveyor_id, given_at) VALUES (?, ?, ?)",
                (respondent_id, surveyor_id, datetime.now()))

    cur.execute("UPDATE inventory SET remaining_items = remaining_items - 1 WHERE surveyor_id = ?", (surveyor_id,))
    conn.commit()
    conn.close()

    return jsonify({"status": "success", "message": "답례품 지급 완료!"})

@app.route("/inventory/<int:surveyor_id>", methods=["GET"])
def inventory_status(surveyor_id):
    conn = db_conn()
    cur = conn.cursor()
    cur.execute("SELECT total_items, remaining_items FROM inventory WHERE surveyor_id=?", (surveyor_id,))
    result = cur.fetchone()
    conn.close()
    if result:
        return jsonify({"total": result[0], "remaining": result[1]})
    else:
        return jsonify({"error": "Surveyor not found or no inventory data"}), 404

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
"""

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py created successfully for PyInstaller.")

app.py created successfully for PyInstaller.


In [43]:
!pyinstaller --onefile app.py

277 INFO: PyInstaller: 6.20.0, contrib hooks: 2026.5
278 INFO: Python: 3.12.13
283 INFO: Platform: Linux-6.6.122+-x86_64-with-glibc2.35
283 INFO: Python environment: /usr
284 INFO: wrote /content/dist/app.spec
292 INFO: Module search paths (PYTHONPATH):
['/env/python',
 '/usr/lib/python312.zip',
 '/usr/lib/python3.12',
 '/usr/lib/python3.12/lib-dynload',
 '/usr/local/lib/python3.12/dist-packages',
 '/usr/lib/python3/dist-packages',
 '/content/dist']
pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html
2298 INFO: checking Analysis
2298 INFO: Building Analysis because Analysis-00.toc is non existent
2298 INFO: Looking for Python shared library...
2346 INFO: Using Python shared library: /lib/x86_64-linux-gnu/libpython3.12.so.1.0
2346 INFO: Running Analysis Analysis-00.toc
2346 INFO: Target bytecode optimization level: 0
2346 INFO: Initializing module dependency graph...
2348 INFO: Initializing module graph hook caches...
2401 IN

ERROR:root:Unexpected exception finding object shape
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_debugpy_repr.py", line 54, in get_shape
    shape = getattr(obj, 'shape', None)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 318, in __get__
    obj = instance._get_current_object()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 519, in _get_current_object
    raise RuntimeError(unbound_message) from None
RuntimeError: Working outside of request context.

This typically means that you attempted to use functionality that needed
an active HTTP request. Consult the documentation on testing for
information about how to avoid this problem.


Aborted by user request.


In [42]:
import subprocess
import time
import requests
import threading

# Change directory to where the 'app' executable is located
%cd /content/dist

# Function to run the Flask app in a separate thread
def run_flask_app():
    # Use subprocess.Popen to run the app in the background
    # stdout and stderr are redirected to avoid blocking the Colab kernel output
    global flask_process
    flask_process = subprocess.Popen(['./app'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    # Optionally, you can read output from the process if needed for debugging
    # for line in flask_process.stdout:
    #     print(f"Flask App Output: {line.decode().strip()}")

# Start the Flask app in a new thread
flask_thread = threading.Thread(target=run_flask_app)
flask_thread.start()

print("Flask app started in the background. Waiting for it to initialize...")
time.sleep(5) # Give the server a few seconds to start up

# Base URL for the Flask application
BASE_URL = "http://127.0.0.1:5000"

# Test the /give_gift endpoint
print("\n--- Testing /give_gift endpoint ---")
gift_data = {
    "respondent_id": 1,
    "surveyor_id": 1
}
try:
    response = requests.post(f"{BASE_URL}/give_gift", json=gift_data)
    print("Give Gift Response Status:", response.status_code)
    print("Give Gift Response Body:", response.json())
except requests.exceptions.ConnectionError as e:
    print(f"Error connecting to Flask app: {e}")
    print("Please ensure the Flask app started successfully.")

# Test the /inventory/<surveyor_id> endpoint
print("\n--- Testing /inventory/<surveyor_id> endpoint ---")
surveyor_id = 1
try:
    response = requests.get(f"{BASE_URL}/inventory/{surveyor_id}")
    print("Inventory Response Status:", response.status_code)
    print("Inventory Response Body:", response.json())
except requests.exceptions.ConnectionError as e:
    print(f"Error connecting to Flask app: {e}")
    print("Please ensure the Flask app started successfully.")

# Clean up: Terminate the Flask app process
if 'flask_process' in globals() and flask_process.poll() is None:
    flask_process.terminate()
    flask_process.wait()
    print("\nFlask app process terminated.")

print("API calls demonstration complete.")

/content/dist
Flask app started in the background. Waiting for it to initialize...

--- Testing /give_gift endpoint ---
Error connecting to Flask app: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /give_gift (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7db0c22c24b0>: Failed to establish a new connection: [Errno 111] Connection refused'))
Please ensure the Flask app started successfully.

--- Testing /inventory/<surveyor_id> endpoint ---
Error connecting to Flask app: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /inventory/1 (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7db0c22bf380>: Failed to establish a new connection: [Errno 111] Connection refused'))
Please ensure the Flask app started successfully.

Flask app process terminated.
API calls demonstration complete.


In [ ]:
!pyinstaller --onefile app.py

69 INFO: PyInstaller: 6.20.0, contrib hooks: 2026.5
70 INFO: Python: 3.12.13
71 INFO: Platform: Linux-6.6.122+-x86_64-with-glibc2.35
72 INFO: Python environment: /usr
72 INFO: wrote /content/dist/app.spec
74 INFO: Module search paths (PYTHONPATH):
['/env/python',
 '/usr/lib/python312.zip',
 '/usr/lib/python3.12',
 '/usr/lib/python3.12/lib-dynload',
 '/usr/local/lib/python3.12/dist-packages',
 '/usr/lib/python3/dist-packages',
 '/content/dist']
pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html
703 INFO: checking Analysis
704 INFO: Building Analysis because Analysis-00.toc is non existent
704 INFO: Looking for Python shared library...
717 INFO: Using Python shared library: /lib/x86_64-linux-gnu/libpython3.12.so.1.0
718 INFO: Running Analysis Analysis-00.toc
718 INFO: Target bytecode optimization level: 0
718 INFO: Initializing module dependency graph...
719 INFO: Initializing module graph hook caches...
729 INFO: Analyzing m

In [ ]:
import subprocess
import time
import requests
import threading

# Change directory to where the 'app' executable is located
%cd /content/dist

# Function to run the Flask app in a separate thread
def run_flask_app():
    global flask_process
    flask_process = subprocess.Popen(['./app'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Start the Flask app in a new thread
flask_thread = threading.Thread(target=run_flask_app)
flask_thread.start()

print("Flask app started in the background. Waiting for it to initialize...")
time.sleep(5) # Give the server a few seconds to start up

# Base URL for the Flask application
BASE_URL = "http://127.0.0.1:5000"

# Test the /give_gift endpoint
print("\n--- Testing /give_gift endpoint ---")
gift_data = {
    "respondent_id": 1,
    "surveyor_id": 1
}
try:
    response = requests.post(f"{BASE_URL}/give_gift", json=gift_data)
    print("Give Gift Response Status:", response.status_code)
    print("Give Gift Response Body:", response.json())
except requests.exceptions.ConnectionError as e:
    print(f"Error connecting to Flask app: {e}")
    print("Please ensure the Flask app started successfully.")

# Test the /inventory/<surveyor_id> endpoint
print("\n--- Testing /inventory/<surveyor_id> endpoint ---")
surveyor_id = 1
try:
    response = requests.get(f"{BASE_URL}/inventory/{surveyor_id}")
    print("Inventory Response Status:", response.status_code)
    print("Inventory Response Body:", response.json())
except requests.exceptions.ConnectionError as e:
    print(f"Error connecting to Flask app: {e}")
    print("Please ensure the Flask app started successfully.")

# Clean up: Terminate the Flask app process
if 'flask_process' in globals() and flask_process.poll() is None:
    flask_process.terminate()
    flask_process.wait()
    print("\nFlask app process terminated.")

print("API calls demonstration complete.")

In [21]:
!pip install pyinstaller # Ensure pyinstaller is installed
!pyinstaller --onefile app.py

70 INFO: PyInstaller: 6.20.0, contrib hooks: 2026.5
70 INFO: Python: 3.12.13
72 INFO: Platform: Linux-6.6.122+-x86_64-with-glibc2.35
72 INFO: Python environment: /usr
72 INFO: wrote /content/app.spec
84 INFO: Module search paths (PYTHONPATH):
['/env/python',
 '/usr/lib/python312.zip',
 '/usr/lib/python3.12',
 '/usr/lib/python3.12/lib-dynload',
 '/usr/local/lib/python3.12/dist-packages',
 '/usr/lib/python3/dist-packages',
 '/content']
pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html
710 INFO: checking Analysis
710 INFO: Building Analysis because Analysis-00.toc is non existent
710 INFO: Looking for Python shared library...
727 INFO: Using Python shared library: /lib/x86_64-linux-gnu/libpython3.12.so.1.0
727 INFO: Running Analysis Analysis-00.toc
727 INFO: Target bytecode optimization level: 0
727 INFO: Initializing module dependency graph...
728 INFO: Initializing module graph hook caches...
748 INFO: Analyzing modules for

In [26]:
# /content/dist 디렉토리로 이동하여 생성된 파일 확인
%cd /content/dist
!ls

/content/dist
app


위 목록에서 `app`이라는 이름의 실행 파일을 확인할 수 있습니다. 이제 이 파일을 다운로드할 수 있습니다.

In [29]:
from google.colab import files

# 생성된 실행 파일 다운로드
files.download('app')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [44]:
# 구글코랩에 ai 넣기: import ai
from google.colab import ai
response = ai.generate_text("대한민국의 수도는?")
response

'대한민국의 수도는 **서울**입니다.\n\n서울은 대한민국의 정치, 경제, 문화, 교육의 중심지이자 가장 큰 도시입니다.'

Colab 노트북은 Google 클라우드 서버에서 코드를 실행하므로 사용 중인 컴퓨터 성능과 관계없이 <a href="#using-accelerated-hardware">GPU 및 TPU</a>를 포함한 Google 하드웨어 성능을 활용할 수 있습니다. 브라우저만 있으면 사용 가능합니다.

예를 들어 <strong>Pandas</strong> 코드의 실행이 완료되기를 기다리고 있는데 더 빠르게 실행하고 싶다면 GPU 런타임으로 전환하고 코드를 변경하지 않고도 가속할 수 있는 <a href="https://rapids.ai/cudf-pandas">RAPIDS cuDF</a>와 같은 라이브러리를 사용할 수 있습니다.

Colab에서 Pandas를 가속하는 방법을 자세히 알아보려면 <a href="https://colab.research.google.com/github/rapidsai-community/showcase/blob/main/getting_started_tutorials/cudf_pandas_colab_demo.ipynb">10분 가이드</a> 또는 <a href="https://colab.research.google.com/github/rapidsai-community/showcase/blob/main/getting_started_tutorials/cudf_pandas_stocks_demo.ipynb">미국 주식 시장 데이터 분석 데모</a>를 참고하세요.

<div class="markdown-google-sans">

## 머신러닝
</div>

Colab을 사용하면 <a href="https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/quickstart/beginner.ipynb">코드 몇 줄만으로</a> 이미지 데이터 세트를 가져오고, 이 데이터 세트로 이미지 분류기를 학습시키며, 모델을 평가할 수 있습니다.

Colab은 다음과 같은 분야의 머신러닝 커뮤니티에서 널리 쓰이고 있습니다.
- TensorFlow 시작하기
- 신경망 개발 및 학습시키기
- TPU로 실험하기
- AI 연구 보급하기
- 튜토리얼 만들기

머신러닝 적용 사례를 보여 주는 Colab 메모장 샘플을 확인하려면 아래 <a href="#machine-learning-examples">머신러닝 예시</a>를 참조하세요.

<div class="markdown-google-sans">

## 추가 리소스

### Colab에서 메모장 사용하기

</div>

- [Colab 개요](/notebooks/basic_features_overview.ipynb)
- [Markdown 가이드](/notebooks/markdown_guide.ipynb)
- [라이브러리 가져오기 및 종속 항목 설치하기](/notebooks/snippets/importing_libraries.ipynb)
- [GitHub에서 노트 저장 및 로드하기](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/colab-github-demo.ipynb)
- [대화형 양식](/notebooks/forms.ipynb)
- [대화형 위젯](/notebooks/widgets.ipynb)

<div class="markdown-google-sans">

<a name="working-with-data"></a>
### 데이터로 작업하기
</div>

- [데이터 로드: 드라이브, 스프레드시트, Google Cloud Storage](/notebooks/io.ipynb)
- [차트: 데이터 시각화하기](/notebooks/charts.ipynb)
- [BigQuery 시작하기](/notebooks/bigquery.ipynb)

<div class="markdown-google-sans">

### 머신러닝

<div>

Google의 온라인 머신러닝 과정을 비롯해 머신러닝과 관련된 일부 노트북입니다. 자세한 내용은 <a href="https://developers.google.com/machine-learning/crash-course/">전체 과정 웹사이트</a>를 참고하세요.
- [Pandas DataFrame 소개](https://colab.research.google.com/github/google/eng-edu/blob/main/ml/cc/exercises/pandas_dataframe_ultraquick_tutorial.ipynb)
- [Pandas를 가속화하는 RAPIDS cuDF 소개](https://nvda.ws/rapids-cudf)
- [cuML의 가속기 모드 시작하기](https://colab.research.google.com/github/rapidsai-community/showcase/blob/main/getting_started_tutorials/cuml_sklearn_colab_demo.ipynb)

<div class="markdown-google-sans">

<a name="using-accelerated-hardware"></a>
### 가속 하드웨어 사용하기
</div>

- [Flax NNX API를 사용하여 MNIST 데이터 세트에서 필기 입력 숫자를 분류하도록 CNN 학습시키기](https://colab.research.google.com/github/google/flax/blob/main/docs_nnx/mnist_tutorial.ipynb)
- [JAX로 이미지 분류를 위한 Vision Transformer&#40;ViT&#41; 학습시키기](https://colab.research.google.com/github/jax-ml/jax-ai-stack/blob/main/docs/source/JAX_Vision_transformer.ipynb)
- [JAX를 사용한 Transformer 언어 모델 기반 텍스트 분류](https://colab.research.google.com/github/jax-ml/jax-ai-stack/blob/main/docs/source/JAX_transformer_text_classification.ipynb)

<div class="markdown-google-sans">

<a name="machine-learning-examples"></a>

### 추천 예시

</div>

- <a href="https://docs.jaxstack.ai/en/latest/JAX_for_LLM_pretraining.html">JAX AI 스택으로 miniGPT 언어 모델 학습시키기</a>
- <a href="https://github.com/google/tunix/blob/main/examples/qlora_gemma.ipynb">Tunix를 사용한 LLM용 LoRA/QLoRA 미세 조정</a>
- <a href="https://keras.io/examples/keras_recipes/parameter_efficient_finetuning_of_gemma_with_lora_and_qlora/">LoRA 및 QLoRA를 사용한 Gemma의 Parameter-Efficient Fine-Tuning&#40;PEFT&#41;</a>
- <a href="https://keras.io/keras_hub/guides/hugging_face_keras_integration/">Hugging Face Transformers 체크포인트 로드</a>
- <a href="https://keras.io/guides/int8_quantization_in_keras/">Keras의 8비트 정수 양자화</a>
- <a href="https://keras.io/examples/keras_recipes/float8_training_and_inference_with_transformer/">간단한 Transformer 모델을 사용한 Float8 학습 및 추론</a>
- <a href="https://keras.io/keras_hub/guides/transformer_pretraining/">KerasHub로 처음부터 Transformer 사전 학습시키기</a>
- <a href="https://keras.io/examples/vision/mnist_convnet/">단순 MNIST 합성곱 신경망</a>
- <a href="https://keras.io/examples/vision/image_classification_from_scratch/">Keras 3를 사용해 처음부터 이미지 분류 구현</a>
- <a href="https://keras.io/keras_hub/guides/classification_with_keras_hub/">KerasHub를 사용한 이미지 분류</a>
